# Knight's Tour as a Search Problem

## 1. Problem Background

I chose the Knight's Tour problem because it sounds cool!  Knight's Tour is a mathematical chess problem where a single Knight moves around a chessboard and tries to visit each square exactly once.  The Knight moves in an L-shape.  Which means that it goes two squares in one direction and one square in a perpendicular direction.  Since each square is a node, and an edge connects two squares if a Knight is allowed to move between them.  

This natural grid structure makes the Knight's Tour a great search problem to dive into.  The algorithm will start with the Knight on one square, where it will then search through possible legal moves while keeping track of which squares the Knight already visited.  A solution is found when a sequence of moves visits every square exactly one time.  This is related to finding a Hamiltonian path because the path has to visit every node in the graph exactly once.  

To demonstrate this, I will use a 5x5 chessboard and place the Knight in the top left corner to begin.  (0, 0).  I will find an open tour solution. There are two tours I read about.  Open and Closed tour.  Closed tour is when the final square that the Knight lands on is one Knight's move away from the starting square.  Open tour is when the final square that the Knight lands on does not matter, this includes the solutions where the Knight is one move away from the starting square.  The search algorithm used will be the Depth-First Search with backtracking.  I will also set up Warnsdorff's heuristic to try more promising first moves.  

## 2. Defining Knight's Tour as a Search Problem

In this problem, a search node represents one partial Knight's Tour.  The state must include the Knight's current square and the squares that have already been visited.  The squares will be represented as a coordinate pair in the following format, (row, column).  Since I am using a 5x5 board, the row and column values will range from 0 to 4.  

There will be a state that includes the list of all visited squares.  There will be a state to show the Knights current location.  The initial state will have (0, 0) in the visited state and the Knight's location state.  

The possible actions are the legal Knight moves from the current square to an unvisited square.  A Knight can move two squares in one direction and one square in a perpendicular direction.  Here is a list of all possible moves.  Two rows up, and one column left or right.  Two columns right, and one row up or down.  Two rows down, and one column left or right.  Two columns left, and one row up or down.  Any move that will place the Knight outside of the 5x5 grid, or places the Knight on a visited square will not be allowed.  

The transition model describes what happens after an action is chosen.  If the Knight moves from its current square to a legal next square, the new state will be updated to the Knight's current position to that next square.  The next square is also added to the path and marked as visited.  

The goal state is the state in which the Knight has visited all 25 squares on the 5x5 board exactly once.  As a reminder, the open tour was chosen so the end state does not have to be exactly one move from the starting position.  

The path cost function can assign a cost of 1 to each Knight move.  I believe the cost is not useful to this problem because the goal of the problem is not optimal path, rather the goal of this solution is making sure that you visit all tiles one time, and depending on the tour type, finding a solution that is either one move away from the start or not. 

## 3. Search Tree Visualization to Depth 2

The first node in this problem is the initial state where the Knight begins at (0, 0) and the only visited square is (0, 0). From this point on the 5x5 board, there are only two valid moves: (1, 2) and (2, 1). The tree below expands both of those depth-1 nodes to depth 2. Each node shows the Knight's current square and the visited path for that partial tour.

```text
Depth 0
(0, 0), visited: [(0, 0)]
|
+-- Depth 1: (1, 2), visited: [(0, 0), (1, 2)]
|   |
|   +-- Depth 2: (0, 4), visited: [(0, 0), (1, 2), (0, 4)]
|   +-- Depth 2: (2, 0), visited: [(0, 0), (1, 2), (2, 0)]
|   +-- Depth 2: (2, 4), visited: [(0, 0), (1, 2), (2, 4)]
|   +-- Depth 2: (3, 1), visited: [(0, 0), (1, 2), (3, 1)]
|   +-- Depth 2: (3, 3), visited: [(0, 0), (1, 2), (3, 3)]
|
+-- Depth 1: (2, 1), visited: [(0, 0), (2, 1)]
    |
    +-- Depth 2: (0, 2), visited: [(0, 0), (2, 1), (0, 2)]
    +-- Depth 2: (1, 3), visited: [(0, 0), (2, 1), (1, 3)]
    +-- Depth 2: (3, 3), visited: [(0, 0), (2, 1), (3, 3)]
    +-- Depth 2: (4, 0), visited: [(0, 0), (2, 1), (4, 0)]
    +-- Depth 2: (4, 2), visited: [(0, 0), (2, 1), (4, 2)]
```

The square (3, 3) appears under both branches because the Knight can reach it from either (1, 2) or (2, 1). These are still different search nodes because the visited path is different in each branch.


## 4. Working Code Example
Below is an implementation of the Knight's Tour using depth-first search with backtracking.  Before trying all of the moves, it will order them using Warnsdorff heuristic.  This tries the square with the fewest future moves first.  The code was written with assistance from OpenAI Codex, based on GPT-5. 

In [1]:
BOARD_SIZE = 5 # a constant for the size of the chessboard
START = (0, 0) # a constant for the starting square of the knight

KNIGHT_MOVES = [
    (-2, -1), (-2, 1), (-1, -2), (-1, 2),
    (1, -2), (1, 2), (2, -1), (2, 1)
] # a list of all possible moves a knight can make on a chessboard

def in_bounds(square): # a function that checks if a square is within the bounds of the chessboard
    row, col = square
    return 0 <= row < BOARD_SIZE and 0 <= col < BOARD_SIZE

def legal_moves(square, visited): # a function that returns a list of all legal moves a knight can make from a given square, excluding squares that have already been visited
    row, col = square
    moves = []

    for row_change, col_change in KNIGHT_MOVES:
        next_square = (row + row_change, col + col_change)

        if in_bounds(next_square) and next_square not in visited:
            moves.append(next_square)

    return moves

def onward_count(square, visited): # a function that counts the number of legal moves from a given square, excluding squares that have already been visited
    new_visited = visited | {square} 
    return len(legal_moves(square, new_visited))

def ordered_moves(square, visited): # a function that returns a list of legal moves from a given square, sorted by the number of onward moves available from each move (Warnsdorff's rule)
    moves = legal_moves(square, visited)
    return sorted(moves, key=lambda move: onward_count(move, visited))

def knight_tour(path, visited): # a recursive function that attempts to find a knight's tour on the chessboard, starting from the given path and visited squares
    if len(path) == BOARD_SIZE * BOARD_SIZE: # if the path contains all squares on the chessboard, return the path as a solution (base case)
        return path

    current_square = path[-1] # get the last square in the path (the current position of the knight)

    for next_square in ordered_moves(current_square, visited): # iterate through the legal moves from the current square, ordered by the number of onward moves available from each move
        result = knight_tour(path + [next_square], visited | {next_square}) # recursively call the knight_tour function with the updated path and visited squares

        if result is not None: # if a solution is found, return it
            return result

    return None # if no solution is found, return None (backtrack)

solution = knight_tour([START], {START}) # start the knight's tour from the starting square and mark it as visited

print("Solution found:", solution is not None) # print whether a solution was found or not
print("Number of squares visited:", len(solution)) # print the number of squares visited in the solution
print(f"Tour path: {solution}") # print the path of the knight's tour

Solution found: True
Number of squares visited: 25
Tour path: [(0, 0), (1, 2), (0, 4), (2, 3), (4, 4), (3, 2), (4, 0), (2, 1), (0, 2), (1, 0), (3, 1), (4, 3), (2, 4), (0, 3), (1, 1), (3, 0), (4, 2), (3, 4), (1, 3), (0, 1), (2, 0), (4, 1), (2, 2), (1, 4), (3, 3)]


## 5. Explanation of the Output
The output of the solutions states true.  That is because there was a solution found.  The solution contained a full list of moves from the start square that touched all of the grid options. Therefore the solution exists, hence the solution found true.  The next print states the number of squares visited.  Since this is a 5x5 board, the total number of visited squares should be 25 since there are 25 tiles in the 5x5 grid.  That would line up with the solution found meaning that there were all 25 tiles touched. 

The Tour Path is the list of the nodes that were visited in the order that they were visited.  There are 25 coordinates in this list since there were 25 visited nodes.  The first value is the row value and the second value is the column value.  No tile in the grid is visited twice.  

This provided solution is a valid open Knight's Tour because all 25 tiles are visited exactly one time.  Since this is an open tour, the solution was not required to be one move away from the start.  

## 6. How the Code Works

This is my favorite part.  So the top of the code starts with two constants.  One for the board size which we determined to be 5x5 and one for the start position which is 0, 0.  (Totally does not need to be a constant lol).  The next piece of the code is a list of all of the possible moves that a knight can make.  This includes all directions.  There will be checks to see if all 8 of these moves are valid inside of the functions later, this is just all of them.  

in_bounds: This function exists to check whether or not a square is within the boundaries of the board size constant.  This exists to test if the move will move the Knight outside of the bounds of the grid.  

legal_moves: This function returns a list of all of the legal moves that a Knight can make from a given square.  This excludes the squares that have already been visited.  There is a for loop that gets the row and the column value from the specific move that the current index is on, then it adds that coordinate to the coordinate that the Knight is currently on.  Then there is a check using the in_bounds function from before on that move to see if that move will be inside of the boundaries or not, and it also checks if that square is visited.  This rules out the visited squares and the moves that would place the Knight out of bounds.  

onward_count: This function counts the number of legal moves from a given square.  It excludes the squares that have already been visited.  This does a check on the square and counts how many moves are legal and returns that value.  The new_visited variable stores the temporary marked candidate square as visited before counting its future moves.  

ordered_moves: This function is where Warnsdorff's heuristic is applied.  First, it calls legal_moves to get the list of legal moves.  Then it sorts that list using onward_count.  The lambda function performs a sort on each possible move by how many legal moves would be available after moving to that given legal location.  A move that has fewer future choices is placed earlier in the list.  This means that it will prioritize the move restricted moves first, thereby preventing the Knight from leaving the harder to reach tiles until the end.  This is not a guaranteed best solution, but it helps the depth first search find a complete tour faster.  In a normal Knight's Tour, some squares are easy to reach because they have many possible Knight moves.  Other squares are harder to reach because they only have a few possible moves.  If the depth first search saves the restricted squares for later, the Knight has the chance of getting trapped.  This would cause the algorithm to backtrack to find the solution.  Warnsdorff's heuristic helps this out by trying the move that leaves the fewest options in the future out first. 

knight_tour: I love this function because it uses recursion.  The base case of this function is a check to see if the path that the knight is on has reached 25.  This means that it should be complete at this point.  (If the presented grid has a solution).  It stores the current square that the Knight is on inside of current_square.  Then it goes through a for loop that iterates based on the legal moves from the current_square.  This is ordered by the number of onward moves available from each move.  Inside of that loop there is a result variable that recursively calls the knight_tour function again while updating the path and visited squares.  Past that, there is a check if there is a solution.  Once there is you return with it.  If not you return with None.  

Finally, you create a variable solution and set it to the result of knight_tour with the value of 5x5. 

## 7. Algorithm Properties

The algorithm I used for the Knight's Tour is depth first search with backtracking. This is a great choice for this because the Knight's Tour is trying to build out one complete path through the board and not all the paths. The search keeps extending the current path until it reaches all 25 squares or gets stuck. If it gets stuck, it will backtrack and try a different move.

For the 5x5 board example, the algorithm is complete as long as it is allowed to continue to backtrack through all of the possible legal moves. Since the board is finite, there are only a limited number of possible paths. If there is a complete Knight's Tour available, it will be found eventually. If a Knight's Tour does not exist, then it will return None. Warnsdorff's heuristic does not remove moves. It only changes the order in which the moves are tried.

The algorithm is not optimal in the normal shortest-path sense because depth first search returns the first complete tour it finds based on the move order. However, for an open Knight's Tour, every valid solution on a 5x5 board visits all 25 squares exactly once, so every complete solution has the same number of visited squares. Because of that, the main goal is finding a valid complete tour, not finding the shortest path.

The worst-case time complexity is exponential. A knight can have up to 8 possible moves. The search depth for an n x n board is n^2 because there are n^2 squares. A simple upper bound for the worst case is O(8^(n^2)). For the 5x5 board example, the maximum depth is 25 squares, or 24 moves after the starting square. The actual search is usually much faster because visited squares reduce the possible moves, and Warnsdorff's heuristic helps avoid bad paths earlier.

The space complexity is O(n^2). The algorithm must keep track of the visited squares on an n x n board. Since the board has n^2 total squares, the visited information can grow to n^2.
GeeksforGeeks gives O(n^2) space for both the recursive backtracking version and the Warnsdorff-style version of Knight’s Tour. The code stores the current path and the visited squares, meaning the memory depends on the number of squares. For my 5x5 board, that means tracking up to 25 squares.

Warnsdorff's heuristic is one of the most common heuristics for Knight's Tour. It chooses the next square with the fewest onward moves. This helps because it considers the most restrictive squares earlier and leaves easier-to-reach squares available for later. The heuristic can make the search much faster in practice, but it does not change the worst-case exponential time complexity. It also does not hurt completeness in this code because backtracking is still built into the algorithm.


## 8. Sources

- Russell, Stuart J., and Peter Norvig. *Artificial Intelligence: A Modern Approach*, 4th US ed. Pearson, 2020. https://aima.cs.berkeley.edu/. Used for the Chapter 3 search algorithm ideas such as depth-first search, completeness, optimality, and complexity.

- Weisstein, Eric W. "Knight's Tour." *MathWorld--A Wolfram Web Resource*. https://mathworld.wolfram.com/KnightsTour.html. Used for general background on the Knight's Tour problem.

- "Knight's tour." *Wikipedia*. https://en.wikipedia.org/wiki/Knight%27s_tour. Accessed June 24, 2026. Used for background on open and closed tours, the Hamiltonian path connection, and Warnsdorff's rule.

- Marateck, Samuel L. "How good is the Warnsdorff's knight's tour heuristic?" *arXiv*, 2008. https://arxiv.org/abs/0803.4321. Used for information about Warnsdorff's rule as a heuristic.

- GeeksforGeeks. "The Knight's Tour." https://www.geeksforgeeks.org/dsa/the-knights-tour-problem/. Used for time and space complexity information for recursive backtracking and Warnsdorff-style Knight's Tour algorithms.

- Code source: OpenAI. OpenAI Codex, based on GPT-5. Used to help write the Python code example for the Knight's Tour depth-first search with backtracking.
